In [31]:
import requests
import pandas as pd
import numpy as np
import re
import sqlite3
import matplotlib.pyplot as plt

apiKey = "RGAPI-767b6861-1028-44c8-9a56-4b9cbb01ada6"
gameName = 'Shadow'
tagLine = 'MC3'
puuid = '4RWZ8j41vOINfON_3oxDLVn4rgUSRYU6UTwRg-YVxWejFdHIFvQb9j-7gL4tGCLzlMgLjYZL4LAbTA'

def get_puuid(summonerId=None, gameName = None, tagLine=None):
    if summonerId is not None:
        link = 'https://americas.api.riotgames.com/lol/summoner/v4/summoners/'
        response = requests.get(link + summonerId + "?api_key=" + apiKey)
        return response.json()['puuid']
    else:
        link = f'https://americas.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{gameName}/{tagLine}?api_key={apiKey}'
        response = requests.get(link)
        return response.json()['puuid']

In [32]:
def get_idtag(puuid=None):
    link = 'https://americas.api.riotgames.com/riot/account/v1/accounts/by-puuid'
    response = requests.get(link + puuid + "?api_key=" + apiKey)
    id = {'gameName': response.json()['gameName'], 'tagLine': response.json()['tagLine']}
    return id

In [33]:
def get_ladder(top=300):
    root = 'https://na1.api.riotgames.com'
    chall = '/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5'
    gm = '/lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5'
    master = '/lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5'

    chall_response = requests.get(root + chall + "?api_key=" + apiKey)
    gm_response = requests.get(root + gm + "?api_key=" + apiKey)
    master_response = requests.get(root + master + "?api_key=" + apiKey)

    chall_df = pd.DataFrame(chall_response.json()['entries'])
    gm_df = pd.DataFrame(gm_response.json()['entries'])
    master_df = pd.DataFrame(master_response.json()['entries'])

    solo_queue_df = pd.concat([chall_df, gm_df, master_df]).reset_index(drop=True)
    solo_queue_df = solo_queue_df.drop(columns= ['rank', 'inactive', 'freshBlood'])
    return solo_queue_df

In [34]:
def get_match_history(puuid=None, start=0, count=20):
    link = f'https://americas.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?'
    params = f'start={start}&count={count}'

    response = requests.get(link + puuid + params + '&api_key=' + apiKey)
    return response.json()

In [35]:
def get_match(matchId=None):
    link = f'https://americas.api.riotgames.com'
    param = f'/lol/match/v5/matches/{matchId}'

    response = requests.get(link + param + '?api_key=' + apiKey)
    return response.json()

# get_match(matchId='NA1_5543059204')

In [36]:
def process_match(match_json, puuid):

    metadata = match_json['metadata']
    matchId = metadata['matchId']
    participants = metadata['participants']

    info = match_json['info']
    players = info['participants']
    teams = info['teams']

    for team in teams:
        bans = team['bans']
        win = team['win']
        teamId = team['teamId']

        objectives = team['objectives']
        baron = objectives['baron']
        grubs = objectives['horde']
        rift_herald = objectives['riftHerald']
        inhibitor = objectives['inhibitor']
        tower = objectives['tower']

In [37]:
def get_av_matches(puuid=None):
    matchIds = get_match_history(puuid=puuid)
    all_matches = []

    for matchId in matchIds:
        game = get_match(matchId=matchId)
        match_df = process_match(game, puuid=puuid)
        all_matches.append(match_df)

    av_df = pd.concat(all_matches, ignore_index=True)
    return av_df

def get_top_matches(ladder, count=20):
    all_matches = []

    for player in ladder.rows:
        player_matches = get_match_history(puuid=player.puuid)
        all_matches.extend(player_matches)

    unique_matches = list(set(all_matches))
    top_df = pd.DataFrame(unique_matches, columns=['matchId'])
    return top_df

In [38]:
my_df = get_av_matches(puuid='4RWZ8j41vOINfON_3oxDLVn4rgUSRYU6UTwRg-YVxWejFdHIFvQb9j-7gL4tGCLzlMgLjYZL4LAbTA')

ladder = get_ladder()
top_df = get_top_matches(ladder)

# my_df
# top_df

ValueError: All objects passed were None